In [16]:
from sklearn.cluster import SpectralClustering
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)


df = pd.read_csv("battery_observations.csv", index_col=0)
meta = pd.read_csv("battery_metadata.csv", index_col=0)
df2 = df.join(meta[["climate_zone", "chemistry"]])
df2 = df2.sort_values(["battery_id", "cycle_index"])
df2 = df2.reset_index()
meta = meta.reset_index()
g = lambda x: int(x.split('-')[-1])
df2["battery_id"] = df2["battery_id"].apply(g)
meta["battery_id"] = meta["battery_id"].apply(g)
battery_ids = list(set(df2["battery_id"]))
materials = list(set(meta["chemistry"]))
climate_zones = list(set(meta["climate_zone"]))
labels = list(set(df2["degradation_status"]))

train_labels = pd.read_csv("train_batteries.csv")["battery_id"].apply(g)
val_labels = pd.read_csv("validation_batteries.csv")["battery_id"].apply(g)

train = df2[df2["battery_id"].isin(train_labels)]
val = df2[df2["battery_id"].isin(val_labels)]


In [17]:

cycle_features = [
    "soc_start_pct",
    "soc_end_pct",
    "depth_of_discharge_pct",
    "high_soc_exposure_fraction",
    "low_soc_exposure_fraction",
    "charge_c_rate",
    "discharge_c_rate",
    "charge_current_a",
    "discharge_current_a",
    "charge_duration_min",
    "discharge_duration_min",
    "energy_discharged_kwh",
    "ambient_temp_c",
    #"pack_temp_c",
    "cooling_active",
    "pack_voltage_mean_v"
]

extended_features = cycle_features + [
    "battery_age_days",
    "time_since_previous_cycle_hours",
    "equivalent_full_cycles",
    "cumulative_high_temp_hours",
    "pack_voltage_charge_v",
    "pack_voltage_discharge_v"
]

features_A = [
    "battery_age_days",
    "equivalent_full_cycles",
]

features_B = features_A + [
    "internal_resistance_mOhm",
]

features_C = features_B + [
    "cumulative_high_temp_hours",
]


In [18]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors, kneighbors_graph


class NystromSpectralClustering:
    """Spectral clustering (kNN graph, symmetric normalised Laplacian)
    with a Nystrom out-of-sample extension."""

    def __init__(self, n_clusters, n_neighbors=30, n_components=None,
                 n_init=10, random_state=None, scale_by_eigenvalue=True):
        self.n_clusters = n_clusters
        self.n_neighbors = n_neighbors
        self.n_components = n_components
        self.n_init = n_init
        self.random_state = random_state
        self.scale_by_eigenvalue = scale_by_eigenvalue

    def fit(self, X):
        n = X.shape[0]
        m = self.n_components or self.n_clusters
        rng = np.random.RandomState(self.random_state)

        # 1. Symmetric kNN affinity graph (same construction as sklearn)
        A = kneighbors_graph(X, self.n_neighbors, mode="connectivity", include_self=True)
        W = (A + A.T)/2

        # 2. Normalised affinity M = D^-1/2 W D^-1/2
        d = np.asarray(W.sum(axis=1)).ravel()
        d_inv_sqrt = 1.0 / np.sqrt(d)
        D_half = sp.diags(d_inv_sqrt)
        M = D_half @ W @ D_half

        # 3. Top-m eigenvectors (largest eigenvalues of M = smallest of the Laplacian)
        eigvals, U = eigsh(M, k=m, which="LA", v0=rng.uniform(-1, 1, n))
        order = np.argsort(eigvals)[::-1]
        eigvals, U = eigvals[order], U[:, order]

        # 4. Embedding Y = D^-1/2 U, then K-means in embedding space
        self.eigvals_ = eigvals
        self.Y_ = U * d_inv_sqrt[:, None]
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, n_init=self.n_init,
                              random_state=self.random_state).fit(self.Y_)
        self.labels_ = self.kmeans_.labels_

        # 5. Neighbour index over the fitted points, used for the extension
        self.nn_ = NearestNeighbors(n_neighbors=self.n_neighbors).fit(X)
        return self

    def transform(self, X_new, batch_size=20_000):
        """Nystrom extension: embed new points into the spectral space."""
        scale = 1.0 / np.maximum(self.eigvals_, 1e-3) if self.scale_by_eigenvalue else 1.0
        out = []
        for start in range(0, X_new.shape[0], batch_size):
            chunk = X_new[start:start + batch_size]
            idx = self.nn_.kneighbors(chunk, return_distance=False)
            # binary weights -> w(x,i)/d(x) = 1/n_neighbors -> mean of neighbours' Y
            out.append(self.Y_[idx].mean(axis=1) * scale)
        return np.vstack(out)

    def predict(self, X_new):
        return self.kmeans_.predict(self.transform(X_new))

In [ ]:
class_names = ["Normal", "Moderate", "Severe", "Critical"]


def map_clusters_to_statuses(clusters, y):
    """Assign each cluster the most frequent training degradation status."""
    df = pd.DataFrame({"cluster": clusters, "status": y})
    return df.groupby("cluster")["status"].agg(lambda s: s.mode().iloc[0]).to_dict()

def run_spectral_clustering(k, X_train, X_val, y_train, y_val,
                            n_neighbors=30, max_fit_samples=10_000, seed=42):
    rng = np.random.RandomState(seed)
    fit_idx = rng.choice(X_train.shape[0],
                         size=min(max_fit_samples, X_train.shape[0]), replace=False)

    model = NystromSpectralClustering(
        n_clusters=k, n_neighbors=n_neighbors, random_state=seed
    ).fit(X_train[fit_idx])

    train_clusters = model.predict(X_train) # all training rows
    val_clusters = model.predict(X_val)

    mapping = map_clusters_to_statuses(train_clusters, y_train)
    val_pred = np.array([mapping[c] for c in val_clusters])

    metrics = {
        "k": k,
        "ARI": adjusted_rand_score(y_val, val_clusters),
        "NMI": normalized_mutual_info_score(y_val, val_clusters),
        "accuracy": accuracy_score(y_val, val_pred),
        "macro_F1": f1_score(y_val, val_pred, labels=class_names,
                             average="macro", zero_division=0),
        "silhouette": silhouette_score(model.Y_, model.labels_),  # in embedding space
    }
    return metrics, val_pred

In [21]:
use_features = features_C
kmeans_cluster_max = 8
kmeans_cluster_candidates = range(2, kmeans_cluster_max + 1) # k does not need to equal the number of degradation classes.
optimize_metric = "ARI" # out of ARI, silhouette, NMI, accuracy, macro_F1

k_idx_mapping = {x:x-2 for x in kmeans_cluster_candidates}

metrics_df = []
val_pred_structs = np.empty(kmeans_cluster_max - 1, dtype=object)

preprocess = Pipeline([
    ("scaler", StandardScaler()),
    ("variance", VarianceThreshold(threshold=0.0))
])
X_train = preprocess.fit_transform(train[use_features])
X_val = preprocess.transform(val[use_features])
y_train = train["degradation_status"].to_numpy()
y_val = val["degradation_status"].to_numpy()

for i,k in enumerate(kmeans_cluster_candidates):
    metrics, val_pred = run_spectral_clustering(k, X_train, X_val, y_train, y_val)
    metrics_df.append(metrics)
    val_pred_structs[i] = val_pred


metrics_df = pd.DataFrame(metrics_df).sort_values(optimize_metric)
print('metrics df:'); print(metrics_df)
best_k = metrics_df["k"].values[0]

val_predictions_best = val_pred_structs[k_idx_mapping[best_k]]

confusion_df = pd.DataFrame(
    confusion_matrix(
        y_val,
        val_predictions_best,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
    )

print("\nValidation confusion matrix:")
print(confusion_df)


print("\nValidation classification report:")
print(classification_report(
    y_val,
    val_predictions_best,
    labels=class_names,
    zero_division=0
))


metrics df:
   k       ARI       NMI  accuracy  macro_F1  silhouette
0  2  0.039873  0.067733  0.432962  0.243596    0.722189
1  3  0.046763  0.086647  0.450372  0.250351    0.706265
5  7  0.116578  0.191811  0.598295  0.442010    0.475556
4  6  0.137713  0.198186  0.605379  0.448089    0.509387
3  5  0.141378  0.200344  0.580405  0.410777    0.531803
6  8  0.145057  0.234634  0.649244  0.485104    0.464000
2  4  0.197504  0.221190  0.580165  0.411259    0.609167

Validation confusion matrix:
                 Predicted Normal  Predicted Moderate  Predicted Severe  \
Actual Normal                8321                   0               317   
Actual Moderate              6558                   0              1989   
Actual Severe                5277                   0              2497   
Actual Critical                17                   0                10   

                 Predicted Critical  
Actual Normal                     0  
Actual Moderate                   0  
Actual Sever